# Kaggle Training Notebook

This notebook runs the full backbone-training workflow on Kaggle, saves reusable artifacts, and finishes with metric, loss, and SR-vs-LR comparisons.


## Strategy

1. Prepare the Kaggle environment and install the repo dependencies.
2. Cache backbone features for comparable mask and pooling strategies.
3. Benchmark deep-only, handcrafted-only, and concatenated representations with shared splits.
4. Run the classical and deep baseline sweeps when needed.
5. Load saved metrics and produce representation, pooling, mask, and SR-vs-LR comparisons.


In [ ]:
!pip install -q pandas seaborn xgboost lightgbm catboost torchgeo


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: c:\Users\ahtrabelsi\Desktop\stage\s2-super-resolution\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

repo_candidates = [
    Path.cwd(),
    Path("/kaggle/working/S2-super-resolution"),
    Path("/kaggle/working/s2-super-resolution"),
]
REPO = None
for cand in repo_candidates:
    if (cand / "pyproject.toml").exists() and (cand / "scripts").exists():
        REPO = cand
        break

if REPO is None:
    REPO = Path("/kaggle/working/S2-super-resolution")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/AhmedTrb/S2-super-resolution.git", str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "torchgeo"], check=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
sns.set_theme(style="whitegrid", context="talk")

print("Working directory:", REPO)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

ImportError: numpy._core.multiarray failed to import

: 

## Kaggle Setup

Clone the repository if needed, install dependencies, and confirm that CUDA is available before launching the experiment sweeps.


In [ ]:
MODEL_SPECS = [
    # CNN
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_ALL_DINO", "resnet50_dino"),
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS", "resnet50_satlas"),

    # Swin
    ("torchgeo:swin_v2_t", "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS", "swin_t_satlas"),
    ("torchgeo:swin_v2_b", "Swin_V2_B_Weights.SENTINEL2_SI_MS_SATLAS", "swin_b_satlas"),

    # ViT
    ("torchgeo:vit_small_patch16_224", "ViTSmall16_Weights.SENTINEL2_ALL_DINO", "vit_small_dino"),
    ("torchgeo:vit_base_patch16_224", "ViTBase16_Weights.SENTINEL2_ALL_MAE", "vit_base_mae"),
    ("torchgeo:vit_base_patch14_dinov2", "ViTBase14_DINOv2_Weights.SENTINEL2_ALL_SOFTCON", "vit_base_softcon"),
]


def make_experiments(resolution):
    return [
        {
            "resolution": resolution,
            "backbone": backbone,
            "weight": weight,
            "run_name": f"{resolution}_{name}",
        }
        for backbone, weight, name in MODEL_SPECS
    ]


SR_EXPERIMENTS = make_experiments("sr")
LR_EXPERIMENTS = make_experiments("lr")
CLASSIC_REGRESSORS = ["ridge", "pls", "random_forest", "extra_trees", "xgboost"]


REPO = Path.cwd()
INVENTORY = REPO / "yellowness_dataset" / "observations_inventory.csv"
DATASET_DIR = REPO / "yellowness_dataset"
ML_ROOT = REPO / "outputs" / "yellowness_backbone_ml"
DEEP_ROOT = REPO / "outputs" / "yellowness_regression"
PLOT_ROOT = REPO / "outputs" / "kaggle_plots"
for path in (ML_ROOT, DEEP_ROOT, PLOT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

python_bin = sys.executable
COMMON_ML_ARGS = {
    "mask_fusion": "feature_mask_pool",
    "dr_methods": ["none", "pca"],
    "pca_variance": "0.95",
    "pls_top_k": "8",
    "batch_size": "16",
    "num_workers": "4",
}

COMMON_DEEP_ARGS = {
    "mask_fusion": "feature_mask_pool",
    "epochs": "40",
    "batch_size": "8",
    "num_workers": "4",
    "learning_rate": "1e-3",
    "backbone_learning_rate": "1e-4",
    "sample_patch_size": "224",
    "center_crop_size": "224",
}


def run_cmd(cmd, allow_failure=True):
    print("RUN:", " ".join(map(str, cmd)))
    try:
        subprocess.run([str(part) for part in cmd], check=True, cwd=REPO)
        return True
    except subprocess.CalledProcessError as exc:
        if not allow_failure:
            raise
        print(f"Skipping failed run: {exc}")
        return False


print(f"Repo: {REPO}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"Inventory: {INVENTORY}")

## Classical Backbone ML Sweep

This section runs the feature-extraction plus classical regression sweep for the full SR and LR experiment matrix.


In [ ]:
# Run the classical backbone-ML sweep end-to-end.
# Set RUN_FULL_SWEEP = False if you only want to inspect the configuration first.
RUN_FULL_SWEEP = True


def run_backbone_ml_experiments(experiments):
    for spec in experiments:
        out_dir = ML_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_backbone_ml.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--pooling", "global_avg",
            "--no-data-parallel",
            "--regressors", *CLASSIC_REGRESSORS,
            "--dr-methods", "none", "pca",
            "--pca-variance", "0.95",
            "--pls-top-k", "8",
            "--batch-size", "16",
            "--num-workers", "4",
            "--save-feature-csv",
            "--feature-csv-name", f"{spec['run_name']}_embeddings.csv",
            "--export-deep-runs-summary",
            "--deep-runs-root", str(DEEP_ROOT),
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)


if RUN_FULL_SWEEP:
    run_backbone_ml_experiments(SR_EXPERIMENTS)
    run_backbone_ml_experiments(LR_EXPERIMENTS)


## Representation Benchmark

This section prioritizes representation quality by caching deep features across mask and pooling variants, then benchmarking deep-only, handcrafted-only, and concatenated features under identical train/validation splits.

Priority order:
- representation choice first: deep-only vs handcrafted-only vs concatenated features
- mask strategy and pooling next: no mask, feature-map masking, masked input, crop-to-parcel bbox
- dimensionality reduction and regressors as secondary baselines


In [ ]:
from IPython.display import Image, display

REPRESENTATION_CACHE_ROOT = ML_ROOT / "representation_cache"
REPRESENTATION_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

REPRESENTATION_MODEL_SPECS = [
    MODEL_SPECS[0],
    MODEL_SPECS[3],
    MODEL_SPECS[8],
    MODEL_SPECS[9],
]
# Replace the shortlist above with MODEL_SPECS for the full backbone sweep.

REPRESENTATION_MASK_POOLING_CONFIGS = [
    {"mask_fusion": "image_only", "pooling": "global_avg"},
    {"mask_fusion": "image_only", "pooling": "global_max"},
    {"mask_fusion": "image_only", "pooling": "gem"},
    {"mask_fusion": "feature_mask_pool", "pooling": "global_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_mean_std"},
    {"mask_fusion": "masked_image", "pooling": "global_avg"},
    {"mask_fusion": "masked_image", "pooling": "global_max"},
    {"mask_fusion": "crop_mask_bbox", "pooling": "global_avg"},
    {"mask_fusion": "crop_mask_bbox", "pooling": "gem"},
]

REPRESENTATION_RESOLUTIONS = ["sr", "lr"]
RUN_REPRESENTATION_FEATURE_EXTRACTION = False
REUSE_REPRESENTATION_FEATURES = True


def representation_run_name(resolution, backbone_name, weight_name, mask_fusion, pooling):
    short_name = weight_name.split(".")[-1].lower()
    short_name = short_name.replace("sentinel2_", "").replace("weights", "")
    short_name = short_name.replace(":", "_").replace("/", "_")
    return f"{resolution}_{backbone_name.split(':')[-1]}_{short_name}_{mask_fusion}_{pooling}"


def run_representation_feature_extraction():
    planned = len(REPRESENTATION_RESOLUTIONS) * len(REPRESENTATION_MODEL_SPECS) * len(REPRESENTATION_MASK_POOLING_CONFIGS)
    print(f"Planned cached feature extraction runs: {planned}")
    for resolution in REPRESENTATION_RESOLUTIONS:
        for backbone, weight, _ in REPRESENTATION_MODEL_SPECS:
            for config in REPRESENTATION_MASK_POOLING_CONFIGS:
                run_name = representation_run_name(resolution, backbone, weight, config["mask_fusion"], config["pooling"])
                out_dir = REPRESENTATION_CACHE_ROOT / run_name
                cmd = [
                    python_bin,
                    "scripts/train_yellowness_backbone_ml.py",
                    "--inventory", str(INVENTORY),
                    "--root-dir", str(DATASET_DIR),
                    "--resolution", resolution,
                    "--backbone", backbone,
                    "--torchgeo-weight", weight,
                    "--mask-fusion", config["mask_fusion"],
                    "--pooling", config["pooling"],
                    "--batch-size", "32",
                    "--num-workers", "4",
                    "--extract-only",
                    "--feature-csv-name", "backbone_embeddings.csv",
                    "--output-dir", str(out_dir),
                ]
                if REUSE_REPRESENTATION_FEATURES:
                    cmd.append("--reuse-cached-features")
                run_cmd(cmd)


if RUN_REPRESENTATION_FEATURE_EXTRACTION:
    run_representation_feature_extraction()
else:
    print(f"Representation cache root: {REPRESENTATION_CACHE_ROOT}")

In [ ]:
REPRESENTATION_BENCH_ROOT = ML_ROOT / "representation_benchmark"
REPRESENTATION_BENCH_SUMMARY = REPRESENTATION_BENCH_ROOT / "representation_benchmark_summary.csv"
RUN_REPRESENTATION_BENCHMARK = False
REP_COMPONENT_GRID = [16, 32, 64, 128, 192, 256]
REP_DR_METHODS = ["none", "pca", "incremental_pca", "truncated_svd", "pls"]
REP_REGRESSORS = [
    "bayesian_ridge",
    "ridge",
    "elastic_net",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
]

if RUN_REPRESENTATION_BENCHMARK:
    cmd = [
        python_bin,
        "scripts/run_yellowness_representation_benchmark.py",
        "--inventory", str(INVENTORY),
        "--embedding-glob", str(REPRESENTATION_CACHE_ROOT / "*" / "backbone_embeddings.csv"),
        "--output-dir", str(REPRESENTATION_BENCH_ROOT),
        "--max-ml-dim", "300",
        "--component-grid", *[str(v) for v in REP_COMPONENT_GRID],
        "--dr-methods", *REP_DR_METHODS,
        "--regressors", *REP_REGRESSORS,
    ]
    run_cmd(cmd, allow_failure=False)

if REPRESENTATION_BENCH_SUMMARY.exists():
    representation_summary = pd.read_csv(REPRESENTATION_BENCH_SUMMARY)
    representation_summary = representation_summary.sort_values(["val_rmse", "val_r2", "val_mae"], ascending=[True, False, True]).reset_index(drop=True)
    display(representation_summary.head(30))
    grouped = (
        representation_summary.groupby(["representation", "mask_fusion", "pooling"], as_index=False)[["val_rmse", "val_r2"]]
        .mean()
        .sort_values(["val_rmse", "val_r2"], ascending=[True, False])
    )
    display(grouped.head(20))
else:
    print(f"Benchmark summary not found yet: {REPRESENTATION_BENCH_SUMMARY}")

REPRESENTATION_PLOTS = [
    "predicted_vs_true_top_configs.png",
    "residuals_top_configs.png",
    "feature_dim_vs_performance.png",
    "representation_comparison.png",
    "mask_strategy_comparison.png",
    "pooling_comparison.png",
    "dr_method_comparison.png",
    "regressor_comparison.png",
]
for plot_name in REPRESENTATION_PLOTS:
    plot_path = REPRESENTATION_BENCH_ROOT / plot_name
    if plot_path.exists():
        print(plot_path)
        display(Image(filename=str(plot_path)))

## Deep Model Training

Run the end-to-end regressor training sweep for the same backbone families. These runs save checkpoints, run configs, and epoch histories that feed the loss-evolution plots at the end.


In [ ]:
# Run the deep-model sweep. This is slower than the classical feature pipeline, but it saves the training history needed for loss plots.

def run_deep_experiments(experiments, epochs=40, batch_size=8, num_workers=4):
    for spec in experiments:
        out_dir = DEEP_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_regressor.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--freeze-backbone",
            "--unfreeze-epoch", "25",
            "--epochs", str(epochs),
            "--batch-size", str(batch_size),
            "--num-workers", str(num_workers),
            "--learning-rate", "1e-3",
            "--backbone-learning-rate", "1e-4",
            "--sample-patch-size", "224",
            "--center-crop-size", "224",
            "--rotate-augment",
            "--data-parallel",
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)

# Uncomment one line at a time if you want to split the run into manageable chunks.
# run_deep_experiments(SR_EXPERIMENTS)
# run_deep_experiments(LR_EXPERIMENTS)


## Load Saved Results

Reload the saved classical-model summaries and deep-training histories before generating the comparison plots.


In [ ]:
def save_fig(fig, filename):
    path = PLOT_ROOT / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return path


def load_ml_results():
    frames = []
    for path in sorted(ML_ROOT.glob("sr_*/summary.csv")) + sorted(ML_ROOT.glob("lr_*/summary.csv")):
        if not path.exists():
            continue
        df = pd.read_csv(path)
        df["run"] = path.parent.name
        df["resolution"] = "sr" if path.parent.name.startswith("sr_") else "lr"
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_deep_history():
    rows = []
    for path in sorted(DEEP_ROOT.glob("**/training_history.json")):
        run_dir = path.parent
        try:
            history = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        meta = {"run": run_dir.name}
        config_path = run_dir / "run_config.json"
        if config_path.exists():
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
                meta.update(
                    {
                        "resolution": config.get("resolution"),
                        "backbone": config.get("backbone"),
                        "torchgeo_weight": config.get("torchgeo_weight"),
                        "mask_fusion": config.get("mask_fusion"),
                    }
                )
            except Exception:
                pass
        for row in history:
            rows.append({**meta, **row})
    return pd.DataFrame(rows)


ml_results = load_ml_results()
deep_history = load_deep_history()

display(ml_results.head())
display(deep_history.head())

In [ ]:
def best_ml_rows(df):
    if df.empty:
        return df
    key_cols = ["resolution", "backbone", "regressor"]
    best_idx = df.groupby(key_cols)["val_rmse"].idxmin()
    return df.loc[best_idx].reset_index(drop=True)


def plot_sr_vs_lr_metrics(df):
    if df.empty:
        print("No ML results found yet.")
        return pd.DataFrame(), pd.DataFrame()

    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "regressor"], as_index=False)[["train_rmse", "val_rmse", "train_mae", "val_mae", "train_r2", "val_r2"]].mean()

    figure_specs = [
        ("train_rmse", "val_rmse", "RMSE"),
        ("train_mae", "val_mae", "MAE"),
        ("train_r2", "val_r2", "R2"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
    for ax, (train_col, val_col, metric_name) in zip(axes, figure_specs):
        melted = agg.melt(
            id_vars=["resolution", "regressor"],
            value_vars=[train_col, val_col],
            var_name="split",
            value_name="value",
        )
        melted["split"] = melted["split"].map({train_col: "train", val_col: "validation"})
        melted["series"] = melted["resolution"].str.upper() + " / " + melted["split"]
        sns.barplot(data=melted, x="regressor", y="value", hue="series", ax=ax, errorbar=None)
        ax.set_title(f"{metric_name}: train vs validation")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_metrics.png")
    print(f"Saved: {saved}")
    plt.show()

    return best, agg


def plot_backbone_comparison(df):
    if df.empty:
        return
    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "backbone"], as_index=False)["val_rmse"].mean()
    fig, ax = plt.subplots(figsize=(18, 6))
    sns.barplot(data=agg, x="backbone", y="val_rmse", hue="resolution", ax=ax, errorbar=None)
    ax.set_title("SR vs LR by backbone (best validation RMSE per run)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_backbones.png")
    print(f"Saved: {saved}")
    plt.show()


def plot_deep_loss_evolution(history_df, top_n=6):
    if history_df.empty:
        print("No deep training history found yet.")
        return

    hist = history_df.copy()
    hist["epoch"] = hist["epoch"].astype(int)
    ranked_runs = (
        hist.groupby(["run", "resolution", "backbone"], as_index=False)["val_loss"]
        .min()
        .sort_values("val_loss")
        .head(top_n)
    )
    selected = hist.merge(ranked_runs[["run"]], on="run", how="inner")

    fig, axes = plt.subplots(len(ranked_runs), 1, figsize=(14, max(4, 4 * len(ranked_runs))), sharex=True)
    if len(ranked_runs) == 1:
        axes = [axes]

    for ax, run_name in zip(axes, ranked_runs["run"].tolist()):
        run_df = selected[selected["run"] == run_name].sort_values("epoch")
        sns.lineplot(data=run_df, x="epoch", y="train_loss", ax=ax, label="train")
        sns.lineplot(data=run_df, x="epoch", y="val_loss", ax=ax, label="validation")
        meta = run_df.iloc[0]
        ax.set_title(f"{meta.get('resolution', '')} | {meta.get('backbone', '')} | {run_name}")
        ax.set_ylabel("Loss")

    plt.tight_layout()
    saved = save_fig(fig, "deep_loss_evolution.png")
    print(f"Saved: {saved}")
    plt.show()


best_ml_results, ml_aggregated = plot_sr_vs_lr_metrics(ml_results)
plot_backbone_comparison(ml_results)
plot_deep_loss_evolution(deep_history)

## Outputs And Saved Artifacts

Each run writes its own model artifacts, metric tables, and figure files.

Saved outputs include:
- `summary.csv` and `summary.json` for per-run metrics
- `model.pkl` for the fitted classical model and preprocessing stack
- `representation_benchmark_summary.csv` plus per-configuration predictions, metrics, and plots for the representation benchmark
- `best_model.pt`, `training_history.json`, and `run_config.json` for deep runs
- `backbone_embeddings.csv` and `backbone_embeddings.npz` for reusable embeddings
- saved plots under `outputs/kaggle_plots/`


In [ ]:
import shutil
from datetime import datetime
from IPython.display import FileLink, display

# Zip all experiment outputs into one archive under /kaggle/working
archive_base = Path("/kaggle/working") / f"yellowness_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
archive_dir = archive_base.parent / archive_base.name
archive_dir.mkdir(parents=True, exist_ok=True)

items_to_collect = [
    REPO / "outputs" / "yellowness_backbone_ml",
    REPO / "outputs" / "yellowness_regression",
    REPO / "outputs" / "kaggle_plots",
]

for item in items_to_collect:
    if item.exists():
        target = archive_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)
    else:
        print(f"Skipped missing output path: {item}")

zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=archive_dir))
print(f"Created archive: {zip_path}")
print("Download from the link below or from Kaggle Files pane (/kaggle/working).")
display(FileLink(str(zip_path)))